# DFS504 — Domain B：Credit Risk & Default Intelligence
## 期末作業報告

| 項目 | 內容 |
|------|------|
| **課程** | DFS504 Big Data & AI in Finance |
| **主題** | Credit Risk & Default Intelligence (Domain B) |
| **資料集** | UCI Credit Card Default Dataset（30,000 筆） |
| **模型** | Decision Tree (Baseline)、Logistic Regression、Random Forest、LightGBM |
| **主要指標** | ROC-AUC（業界標準信用評分指標） |

## 1. 研究動機與資料說明

### 問題定義
信用卡違約預測是金融機構最核心的風控挑戰之一。正確識別高違約風險客戶，能幫助銀行：
- 降低信用損失
- 提前介入高風險帳戶
- 優化授信政策

### 資料集
- **來源**：UCI Machine Learning Repository — Default of Credit Card Clients
- **大小**：30,000 筆 × 23 個特徵
- **目標變數**：`default.payment.next.month`（0 = 未違約，1 = 違約）
- **類別不平衡**：違約率約 22%，須特別處理

### 為何選 ROC-AUC 為主要指標？
- 資料不平衡時 Accuracy 具誤導性（全猜 0 就有 78% 準確率）
- ROC-AUC 衡量所有門檻下模型的辨別能力
- 業界信用評分標準（Gini 係數 = 2 × AUC − 1）

In [ ]:
# ── 套件與常數 ────────────────────────────────────────────────────────────────
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, average_precision_score,
    confusion_matrix, classification_report, precision_recall_curve,
)

DATA_PATH    = r'C:\Users\Acer\AppData\Local\Temp\UCI_Credit_Card.csv'
TARGET       = 'default.payment.next.month'
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print('✅ 套件載入完成')

## 2. 資料載入與探索分析 (EDA)

In [ ]:
# ── 載入資料 ──────────────────────────────────────────────────────────────────
df_raw = pd.read_csv(DATA_PATH)
df_raw.drop(columns=['ID'], inplace=True)
df_raw['EDUCATION'] = df_raw['EDUCATION'].replace({0: 4, 5: 4, 6: 4})
df_raw['MARRIAGE']  = df_raw['MARRIAGE'].replace({0: 3})

print(f'Shape    : {df_raw.shape}')
print(f'違約率   : {df_raw[TARGET].mean():.2%}  (class imbalance present)')
print(f'缺失值   : {df_raw.isnull().sum().sum()}')
df_raw.head(3)

In [ ]:
# ── EDA 視覺化 ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.suptitle('Exploratory Data Analysis', fontsize=14, fontweight='bold')

counts = df_raw[TARGET].value_counts()
axes[0].bar(['No Default', 'Default'], counts.values,
            color=['steelblue', 'tomato'], edgecolor='black')
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 200, f'{v:,}\n({v/len(df_raw):.1%})', ha='center', fontsize=10)

for label, color in [(0, 'steelblue'), (1, 'tomato')]:
    lbl = 'No Default' if label == 0 else 'Default'
    axes[1].hist(df_raw[df_raw[TARGET]==label]['LIMIT_BAL'],
                 bins=40, alpha=0.6, color=color, label=lbl)
axes[1].set_title('Credit Limit Distribution')
axes[1].set_xlabel('LIMIT_BAL (NT$)')
axes[1].legend()

for label, color in [(0, 'steelblue'), (1, 'tomato')]:
    lbl = 'No Default' if label == 0 else 'Default'
    axes[2].hist(df_raw[df_raw[TARGET]==label]['AGE'],
                 bins=30, alpha=0.6, color=color, label=lbl)
axes[2].set_title('Age Distribution')
axes[2].set_xlabel('Age')
axes[2].legend()

plt.tight_layout()
plt.show()

## 3. Feature Engineering

在原始 23 個特徵外，新增 6 個衍生特徵：

| 特徵名稱 | 計算方式 | 意義 |
|---------|---------|------|
| `utilization_rate` | avg_bill / LIMIT_BAL | 信用使用率（越高越危險）|
| `avg_pay_ratio` | avg_pay / avg_bill | 平均還款比率（越低越危險）|
| `consecutive_delay` | count(PAY_x > 0) | 連續遲繳月數 |
| `bill_trend` | BILL_AMT1 − BILL_AMT6 | 帳單趨勢（正值 = 帳單增加）|
| `pay_trend` | PAY_AMT1 − PAY_AMT6 | 繳款趨勢（正值 = 繳款增加）|
| `credit_age_ratio` | LIMIT_BAL / AGE | 年齡信用比（信用成熟度）|

In [ ]:
# ── Feature Engineering ───────────────────────────────────────────────────────
df = df_raw.copy()

bill_cols  = ['BILL_AMT1','BILL_AMT2','BILL_AMT3','BILL_AMT4','BILL_AMT5','BILL_AMT6']
pay_cols   = ['PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']
delay_cols = ['PAY_0','PAY_2','PAY_3','PAY_4','PAY_5','PAY_6']

avg_bill = df[bill_cols].mean(axis=1)
avg_pay  = df[pay_cols].mean(axis=1)

df['utilization_rate']  = np.where(df['LIMIT_BAL'] > 0, avg_bill / df['LIMIT_BAL'], 0).clip(0, 5)
df['avg_pay_ratio']     = np.where(avg_bill > 0, avg_pay / avg_bill, 1.0).clip(0, 5)
df['consecutive_delay'] = (df[delay_cols] > 0).sum(axis=1)
df['bill_trend']        = df['BILL_AMT1'] - df['BILL_AMT6']
df['pay_trend']         = df['PAY_AMT1']  - df['PAY_AMT6']
df['credit_age_ratio']  = df['LIMIT_BAL'] / df['AGE']

new_feats = ['utilization_rate','avg_pay_ratio','consecutive_delay',
             'bill_trend','pay_trend','credit_age_ratio']
print(f'特徵總數: {df_raw.shape[1]-1} 原始 + {len(new_feats)} 新增 = {df.shape[1]-1} 個特徵')
df[new_feats].describe().round(3)

## 4. 資料分割與前處理

In [ ]:
# ── Train / Test Split（8:2，Stratified）────────────────────────────────────
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# StandardScaler（僅供 Logistic Regression 使用）
cat_cols = ['SEX','EDUCATION','MARRIAGE','PAY_0','PAY_2','PAY_3','PAY_4','PAY_5','PAY_6']
num_cols = [c for c in X_train.columns if c not in cat_cols]

scaler    = StandardScaler()
X_train_s = X_train.copy()
X_test_s  = X_test.copy()
X_train_s[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_s[num_cols]  = scaler.transform(X_test[num_cols])

print(f'Train : {X_train.shape}  |  Test : {X_test.shape}')
print(f'Train default rate : {y_train.mean():.2%}')
print(f'Test  default rate : {y_test.mean():.2%}')

In [ ]:
# ── 評估輔助函數 ───────────────────────────────────────────────────────────────
def evaluate_model(name, model, X_te, y_te):
    y_pred  = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]
    return {
        'Model':         name,
        'Accuracy':      round(accuracy_score(y_te, y_pred), 4),
        'Precision':     round(precision_score(y_te, y_pred), 4),
        'Recall':        round(recall_score(y_te, y_pred), 4),
        'F1':            round(f1_score(y_te, y_pred), 4),
        'ROC-AUC':       round(roc_auc_score(y_te, y_proba), 4),
        'Avg Precision': round(average_precision_score(y_te, y_proba), 4),
        'y_proba':       y_proba,
        'y_pred':        y_pred,
    }


def plot_eval(result, y_te, color, axes4, feat_df=None, feat_color='gray'):
    cm = confusion_matrix(y_te, result['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', ax=axes4[0],
                xticklabels=['No Default','Default'],
                yticklabels=['No Default','Default'])
    axes4[0].set_title('Confusion Matrix')
    axes4[0].set_ylabel('Actual'); axes4[0].set_xlabel('Predicted')

    fpr, tpr, _ = roc_curve(y_te, result['y_proba'])
    axes4[1].plot(fpr, tpr, color=color, lw=2, label=f"AUC={result['ROC-AUC']:.4f}")
    axes4[1].plot([0,1],[0,1],'k--', lw=1)
    axes4[1].set_title('ROC Curve')
    axes4[1].set_xlabel('FPR'); axes4[1].set_ylabel('TPR')
    axes4[1].legend(); axes4[1].grid(alpha=0.3)

    prec, rec, _ = precision_recall_curve(y_te, result['y_proba'])
    axes4[2].step(rec, prec, color=color, lw=2, label=f"AP={result['Avg Precision']:.4f}")
    axes4[2].axhline(y=y_te.mean(), color='gray', linestyle='--', label='Baseline')
    axes4[2].set_title('Precision-Recall Curve')
    axes4[2].set_xlabel('Recall'); axes4[2].set_ylabel('Precision')
    axes4[2].legend(); axes4[2].grid(alpha=0.3)

    if feat_df is not None:
        top = feat_df.head(15)
        axes4[3].barh(range(len(top)), top['Importance'].values,
                      color=feat_color, edgecolor='black')
        axes4[3].set_yticks(range(len(top)))
        axes4[3].set_yticklabels(top['Feature'].values, fontsize=9)
        axes4[3].set_title('Feature Importance (Top 15)')
        axes4[3].invert_yaxis()


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print('✅ 評估函數定義完成')

---
## 5. Baseline Model — Decision Tree

**選擇理由**：決策樹為最直觀的樹狀分類器，規則明確可解釋，適合作為基準比較。  
**參數設定**：`max_depth=5`（防止過擬合），`class_weight='balanced'`（處理類別不平衡）

In [ ]:
# ── Decision Tree — 5-Fold CV + 測試集 ───────────────────────────────────────
dt = DecisionTreeClassifier(max_depth=5, class_weight='balanced', random_state=RANDOM_STATE)

cv_dt = cross_validate(dt, X_train, y_train, cv=skf,
                       scoring=['accuracy','precision','recall','f1','roc_auc'], n_jobs=-1)
print('=== Decision Tree (Baseline) — 5-Fold CV ===')
for m in ['accuracy','precision','recall','f1','roc_auc']:
    print(f'  {m:<12}: {cv_dt[f"test_{m}"].mean():.4f} ± {cv_dt[f"test_{m}"].std():.4f}')

dt.fit(X_train, y_train)
res_dt = evaluate_model('Decision Tree (Baseline)', dt, X_test, y_test)

print('\n=== Test Set Results ===')
for k in ['Accuracy','Precision','Recall','F1','ROC-AUC','Avg Precision']:
    print(f'  {k:<15}: {res_dt[k]}')
print('\n', classification_report(y_test, res_dt['y_pred'], target_names=['No Default','Default']))

In [ ]:
# ── Decision Tree — 視覺化 ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(24, 5))
fig.suptitle('Decision Tree (Baseline) — Evaluation', fontsize=14, fontweight='bold')
fi_dt = pd.DataFrame({'Feature': X_train.columns, 'Importance': dt.feature_importances_})
fi_dt = fi_dt.sort_values('Importance', ascending=False)
plot_eval(res_dt, y_test, 'darkorange', axes, feat_df=fi_dt, feat_color='darkorange')
plt.tight_layout()
plt.show()

---
## 6. Advanced Model 1 — Logistic Regression

**選擇理由**：線性模型，係數直接對應各特徵影響方向，業界信用評分廣泛採用，可解釋性強。  
**參數設定**：`max_iter=1000`，`class_weight='balanced'`，特徵先以 StandardScaler 標準化。

In [ ]:
# ── Logistic Regression — 5-Fold CV + 測試集 ─────────────────────────────────
lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE)

cv_lr = cross_validate(lr, X_train_s, y_train, cv=skf,
                       scoring=['accuracy','precision','recall','f1','roc_auc'], n_jobs=-1)
print('=== Logistic Regression — 5-Fold CV ===')
for m in ['accuracy','precision','recall','f1','roc_auc']:
    print(f'  {m:<12}: {cv_lr[f"test_{m}"].mean():.4f} ± {cv_lr[f"test_{m}"].std():.4f}')

lr.fit(X_train_s, y_train)
res_lr = evaluate_model('Logistic Regression', lr, X_test_s, y_test)

print('\n=== Test Set Results ===')
for k in ['Accuracy','Precision','Recall','F1','ROC-AUC','Avg Precision']:
    print(f'  {k:<15}: {res_lr[k]}')
print('\n', classification_report(y_test, res_lr['y_pred'], target_names=['No Default','Default']))

In [ ]:
# ── Logistic Regression — 視覺化 ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(24, 5))
fig.suptitle('Logistic Regression — Evaluation', fontsize=14, fontweight='bold')
coef_df = pd.DataFrame({
    'Feature':    X_train.columns,
    'Importance': np.abs(lr.coef_[0])
}).sort_values('Importance', ascending=False)
plot_eval(res_lr, y_test, 'steelblue', axes, feat_df=coef_df, feat_color='steelblue')
plt.tight_layout()
plt.show()

---
## 7. Advanced Model 2 — Random Forest

**選擇理由**：集成 200 棵決策樹（Bagging），大幅降低過擬合，ROC-AUC 在本次實驗中最高（0.7763）。  
**參數設定**：`n_estimators=200`，`max_depth=10`，`class_weight='balanced'`

In [ ]:
# ── Random Forest — 5-Fold CV + 測試集 ───────────────────────────────────────
rf = RandomForestClassifier(
    n_estimators=200, max_depth=10,
    class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1
)

cv_rf = cross_validate(rf, X_train, y_train, cv=skf,
                       scoring=['accuracy','precision','recall','f1','roc_auc'], n_jobs=-1)
print('=== Random Forest — 5-Fold CV ===')
for m in ['accuracy','precision','recall','f1','roc_auc']:
    print(f'  {m:<12}: {cv_rf[f"test_{m}"].mean():.4f} ± {cv_rf[f"test_{m}"].std():.4f}')

rf.fit(X_train, y_train)
res_rf = evaluate_model('Random Forest', rf, X_test, y_test)

print('\n=== Test Set Results ===')
for k in ['Accuracy','Precision','Recall','F1','ROC-AUC','Avg Precision']:
    print(f'  {k:<15}: {res_rf[k]}')
print('\n', classification_report(y_test, res_rf['y_pred'], target_names=['No Default','Default']))

In [ ]:
# ── Random Forest — 視覺化 ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(24, 5))
fig.suptitle('Random Forest — Evaluation', fontsize=14, fontweight='bold')
fi_rf = pd.DataFrame({'Feature': X_train.columns, 'Importance': rf.feature_importances_})
fi_rf = fi_rf.sort_values('Importance', ascending=False)
plot_eval(res_rf, y_test, 'forestgreen', axes, feat_df=fi_rf, feat_color='forestgreen')
plt.tight_layout()
plt.show()

---
## 8. Advanced Model 3 — LightGBM

**選擇理由**：現代 Gradient Boosting 框架，訓練速度快，Recall 最高（0.6414），最善於識別真正違約客戶。  
**參數設定**：`n_estimators=200`，`max_depth=6`，`learning_rate=0.05`，`class_weight='balanced'`

In [ ]:
# ── LightGBM — 5-Fold CV + 測試集 ────────────────────────────────────────────
lgbm = lgb.LGBMClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.05,
    class_weight='balanced', random_state=RANDOM_STATE,
    n_jobs=-1, verbose=-1
)

cv_lgbm = cross_validate(lgbm, X_train, y_train, cv=skf,
                         scoring=['accuracy','precision','recall','f1','roc_auc'], n_jobs=-1)
print('=== LightGBM — 5-Fold CV ===')
for m in ['accuracy','precision','recall','f1','roc_auc']:
    print(f'  {m:<12}: {cv_lgbm[f"test_{m}"].mean():.4f} ± {cv_lgbm[f"test_{m}"].std():.4f}')

lgbm.fit(X_train, y_train)
res_lgbm = evaluate_model('LightGBM', lgbm, X_test, y_test)

print('\n=== Test Set Results ===')
for k in ['Accuracy','Precision','Recall','F1','ROC-AUC','Avg Precision']:
    print(f'  {k:<15}: {res_lgbm[k]}')
print('\n', classification_report(y_test, res_lgbm['y_pred'], target_names=['No Default','Default']))

In [ ]:
# ── LightGBM — 視覺化 ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(24, 5))
fig.suptitle('LightGBM — Evaluation', fontsize=14, fontweight='bold')
fi_lgbm = pd.DataFrame({'Feature': X_train.columns, 'Importance': lgbm.feature_importances_})
fi_lgbm = fi_lgbm.sort_values('Importance', ascending=False)
plot_eval(res_lgbm, y_test, 'mediumpurple', axes, feat_df=fi_lgbm, feat_color='mediumpurple')
plt.tight_layout()
plt.show()

---
## 9. 模型比較

In [ ]:
# ── 完整比較表 ────────────────────────────────────────────────────────────────
keys = ['Model','Accuracy','Precision','Recall','F1','ROC-AUC','Avg Precision']
comp_df = pd.DataFrame(
    [{k: r[k] for k in keys} for r in [res_dt, res_lr, res_rf, res_lgbm]]
).set_index('Model')

print('=== 模型比較表（Test Set）===')
display(
    comp_df.style
    .highlight_max(axis=0, color='lightgreen')
    .highlight_min(axis=0, color='#ffcccc')
    .format('{:.4f}')
    .set_caption('綠色 = 最高值 | 紅色 = 最低值')
)

In [ ]:
# ── ROC + PR 曲線比較 ─────────────────────────────────────────────────────────
palette = [
    (res_dt,   'darkorange',   'Decision Tree (Baseline)'),
    (res_lr,   'steelblue',    'Logistic Regression'),
    (res_rf,   'forestgreen',  'Random Forest'),
    (res_lgbm, 'mediumpurple', 'LightGBM'),
]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('All Models Comparison', fontsize=14, fontweight='bold')

for res, color, name in palette:
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    axes[0].plot(fpr, tpr, color=color, lw=2,
                 label=f"{name} (AUC={res['ROC-AUC']:.4f})")
axes[0].plot([0,1],[0,1],'k--', lw=1)
axes[0].set_title('ROC Curves')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(loc='lower right', fontsize=9)
axes[0].grid(alpha=0.3)

for res, color, name in palette:
    prec, rec, _ = precision_recall_curve(y_test, res['y_proba'])
    axes[1].step(rec, prec, color=color, lw=2,
                 label=f"{name} (AP={res['Avg Precision']:.4f})")
axes[1].axhline(y=y_test.mean(), color='gray', linestyle='--', label='No-skill baseline')
axes[1].set_title('Precision-Recall Curves')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend(loc='upper right', fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── 三大指標長條圖 ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Metric Comparison Across Models', fontsize=14, fontweight='bold')

short_names = ['DT\n(Baseline)', 'LR', 'RF', 'LGBM']
colors      = ['darkorange', 'steelblue', 'forestgreen', 'mediumpurple']

for ax, metric in zip(axes, ['ROC-AUC', 'Recall', 'F1']):
    vals = [r[0][metric] for r in palette]
    bars = ax.bar(short_names, vals, color=colors, edgecolor='black', alpha=0.85)
    ax.set_title(f'{metric} Comparison')
    ax.set_ylabel(metric)
    ax.set_ylim(0, 1)
    ax.axhline(y=vals[0], color='darkorange', linestyle='--', alpha=0.5, label='Baseline level')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.01,
                f'{v:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## 10. 結論

### 模型效能摘要

| 排名 | 模型 | ROC-AUC | 主要優勢 | 適用場景 |
|------|------|--------:|---------|----------|
| 📌 Baseline | Decision Tree | 0.7543 | 可解釋性強，規則明確 | 教學與解釋用途 |
| 🥇 1 | **Random Forest** | **0.7763** | AUC 最高，綜合最穩 | 生產部署首選 |
| 🥈 2 | LightGBM | 0.7643 | Recall 最高，漏抓最少 | 高風險偵測場景 |
| 🥉 3 | Logistic Regression | 0.7436 | 線性可解釋，係數直觀 | 法規合規場景 |

### 關鍵發現

1. **ROC-AUC 為主要指標**：資料違約率僅 22%，Accuracy 具誤導性。ROC-AUC 衡量所有門檻下的辨別能力，為業界信用評分標準（等價 Gini 係數）。

2. **集成模型優於單模型**：Random Forest 與 LightGBM 均優於 Baseline Decision Tree，說明 Bagging / Boosting 策略能有效提升預測力。

3. **最重要特徵**：`PAY_0`（最近一月繳款狀態）、`consecutive_delay`（連續遲繳月數）、`utilization_rate`（信用使用率）在各模型中一致排名前列。

4. **Precision vs Recall 權衡**：
   - 減少信用損失（不能漏抓）→ 選 **LightGBM**（Recall 最高）
   - 減少誤判優質客戶 → 選 **Logistic Regression**（Precision 相對穩定）
   - 整體最佳 → 選 **Random Forest**（AUC 最高）

### 未來改進方向
- **閾值最佳化**：將預測門檻由 0.5 調整至 0.3~0.4 以進一步提升 Recall
- **SMOTE 過採樣**或 cost-sensitive learning 處理類別不平衡
- **SHAP 解釋性分析**：提供逐一客戶的違約原因解讀，符合業界監管要求